# Nicheformer - essentials
----------------
The next cells contain the parts of the original Nicheformer code needed for inference, so the package does not need to be installed:

- `complete_masking`: token masking. With `p=0` (inference) it only replaces 0 with the padding token and builds the attention mask.
- `sf_normalize`, `NicheformerDataset`, `_sub_tokenize_data`: normalization and tokenization of an AnnData (genes ranked by normalized expression).
- `Nicheformer`, `PositionalEncoding`, `CosineWarmupScheduler`: the model (Lightning module) and its helpers.

In [1]:
import torch
import torch.nn as nn
import torch.nn.init as init
import pytorch_lightning as pl
from typing import List, Dict, Optional, Union
from torch import optim
import numpy as np
import math


def complete_masking(batch, p, n_tokens):
    
    padding_token = 1
    cls_token = 3

    indices = batch['X']

    indices = torch.where(indices == 0, torch.tensor(padding_token), indices) # 0 is originally the padding token, we change it to 1
    batch['X'] = indices

    mask = 1 - torch.bernoulli(torch.ones_like(indices), p) # mask indices with probability p
    
    masked_indices = indices * mask # masked_indices 
    masked_indices = torch.where(indices != padding_token, masked_indices, indices) # we just mask non-padding indices
    mask = torch.where(indices == padding_token, torch.tensor(padding_token), mask) # in the model we evaluate the loss of mask position 0
    # so we make the mask of all PAD tokens to be 1 so that it's not taken into account in the loss computation
    
    # Notice for the following 2 lines that masked_indices has already not a single padding token masked
    masked_indices = torch.where(indices != cls_token, masked_indices, indices) # same with CLS, no CLS token can be masked
    mask = torch.where(indices == cls_token, torch.tensor(padding_token), mask) # we change the mask so that it doesn't mask any CLS token

    # 80% of masked indices are masked
    # 10% of masked indices are a random token
    # 10% of masked indices are the real token

    random_tokens = torch.randint(10, n_tokens, size=masked_indices.shape, device=masked_indices.device)
    random_tokens = random_tokens * torch.bernoulli(torch.ones_like(random_tokens)*0.1).type(torch.int64) 

    masked_indices = torch.where(masked_indices == 0, random_tokens, masked_indices) # put random tokens just in the previously masked tokens

    same_tokens = indices.clone()
    same_tokens = same_tokens * torch.bernoulli(torch.ones_like(same_tokens) * 0.1).type(torch.int64)

    masked_indices = torch.where(masked_indices == 0, same_tokens, masked_indices) # put same tokens just in the previously masked tokens

    batch['masked_indices'] = masked_indices
    batch['mask'] = mask
    
    attention_mask = (masked_indices == padding_token)
    batch['attention_mask'] = attention_mask.type(torch.bool)

    return batch


MASK_TOKEN = 0
CLS_TOKEN = 2


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import gc
import numpy as np
import torch
from anndata import AnnData
from scipy.sparse import issparse
from tqdm import tqdm
from torch.utils.data import Dataset
from sklearn.utils import sparsefuncs
import numba

def sf_normalize(X):
    """Normalize the input matrix to a scale of 10000."""
    X = X.copy()
    counts = np.array(X.sum(axis=1))
    # avoid division by zero for empty cells
    counts += counts == 0.
    # normalize to 10000. counts
    scaling_factor = 10000. / counts

    if issparse(X):
        sparsefuncs.inplace_row_scale(X, scaling_factor)
    else:
        np.multiply(X, scaling_factor.reshape((-1, 1)), out=X)

    return X

class NicheformerDataset(Dataset):
    """Dataset for Nicheformer"""

    def __init__(self, adata, technology_mean, split='train', max_seq_len=4096, aux_tokens=30, chunk_size=1000,
                 metadata_fields=None):
        """
        Initialize the dataset

        Args:
            adata (AnnData): Annotated data matrix
            technology_mean (np.array): technology mean
            split (str): 'train', 'test', or 'val'
            max_seq_len (int): Maximum sequence length for tokenization
            aux_tokens (int): Number of reserved tokens
            chunk_size (int): Number of cells to process at once
            metadata_fields (dict): Dictionary specifying which metadata fields to include.
                                  Format: {
                                      'obs': ['field1', 'field2'],  # fields from adata.obs
                                      'obsm': ['field3', 'field4']  # fields from adata.obsm
                                  }
        """
        self.adata = adata[adata.obs.nicheformer_split == split].copy()
        self.technology_mean = technology_mean
        self.max_seq_len = max_seq_len
        self.aux_tokens = aux_tokens
        self.chunk_size = chunk_size
        self.metadata_fields = metadata_fields or {'obs': [], 'obsm': []}

        # Initialize storage for tokenized data
        self.n_cells = len(self.adata)
        self.tokens = None

        # Process data in chunks
        self._process_chunks()

        # Store metadata
        self._prepare_metadata()

    def _process_chunks(self):
        n_chunks = (self.n_cells + self.chunk_size - 1) // self.chunk_size
        tokens_list = []

        for chunk_idx in tqdm(range(n_chunks)):
            start_idx = chunk_idx * self.chunk_size
            end_idx = min((chunk_idx + 1) * self.chunk_size, self.n_cells)

            # Process chunk
            chunk_tokens = self._process_chunk(start_idx, end_idx)
            tokens_list.append(chunk_tokens)

            torch.cuda.empty_cache()
            gc.collect()  # force garbage collection

        # Concatenate all chunks
        self.tokens = np.concatenate(tokens_list, axis=0)

    def _process_chunk(self, start_idx, end_idx):
        # Get chunk of data
        chunk_adata = self.adata[start_idx:end_idx]

        # Convert sparse to dense for this chunk only
        if issparse(chunk_adata.X):
            x = chunk_adata.X.toarray()
        else:
            x = chunk_adata.X

        # Process chunk
        x = np.nan_to_num(x)
        x = sf_normalize(x)

        tech_mean = self.technology_mean
        tech_mean += tech_mean == 0
        x = x / tech_mean.reshape((1, -1))

        # Tokenize
        tokens = _sub_tokenize_data(x, self.max_seq_len, self.aux_tokens).astype(np.int32)

        return tokens

    def _prepare_metadata(self):
        self.metadata = {}

        # Process obs fields - direct assignment, no special processing needed
        for field in self.metadata_fields.get('obs', []):
            self.metadata[field] = self.adata.obs[field].values

        # Process obsm fields - chunk processing only if sparse
        for field in self.metadata_fields.get('obsm', []):
            if issparse(self.adata.obsm[field]):
                vectors = []
                for chunk_idx in range(0, self.n_cells, self.chunk_size):
                    end_idx = min(chunk_idx + self.chunk_size, self.n_cells)
                    chunk = self.adata.obsm[field][chunk_idx:end_idx].toarray()
                    vectors.append(chunk)
                self.metadata[field] = np.concatenate(vectors, axis=0)
            else:
                self.metadata[field] = self.adata.obsm[field]

    def __len__(self):
        return self.n_cells

    def __getitem__(self, idx):
        item = {
            'X': torch.tensor(self.tokens[idx])
        }

        # Add all metadata fields to the item
        for key, value in self.metadata.items():
            item[key] = torch.tensor(value[idx])

        return item

In [3]:
@numba.jit(nopython=True, nogil=True)
def _sub_tokenize_data(x: np.array, max_seq_len: int = -1, aux_tokens: int = 30):
    """Tokenize the input gene vector"""
    scores_final = np.empty((x.shape[0], max_seq_len if max_seq_len > 0 else x.shape[1]))
    for i, cell in enumerate(x):
        nonzero_mask = np.nonzero(cell)[0]
        sorted_indices = nonzero_mask[np.argsort(-cell[nonzero_mask])][:max_seq_len] 
        sorted_indices = sorted_indices + aux_tokens # we reserve some tokens for padding etc (just in case)
        if max_seq_len:
            scores = np.zeros(max_seq_len, dtype=np.int32)
        else:
            scores = np.zeros_like(cell, dtype=np.int32)
        scores[:len(sorted_indices)] = sorted_indices.astype(np.int32)

        scores_final[i, :] = scores

    return scores_final

In [4]:
import torch.jit as jit

class Nicheformer(pl.LightningModule):
    def __init__(self,
                 dim_model: int,
                 nheads: int,
                 dim_feedforward: int,
                 nlayers: int,
                 dropout: float,
                 batch_first: bool,
                 masking_p: float,
                 n_tokens: int,
                 context_length: int,
                 lr: float,
                 warmup: int,
                 batch_size: int,
                 max_epochs: int,
                 cls_classes: int = 164,
                 supervised_task: Optional[str] = None,
                 learnable_pe: bool = True,
                 specie: bool = False,
                 assay: bool = False,
                 modality: bool = False,
                 contrastive: bool = False):
        """Initialize NicheformerBase.

        Args:
            dim_model: Dimensionality of the model
            nheads: Number of attention heads
            dim_feedforward: Dimensionality of MLPs in attention blocks
            nlayers: Number of transformer layers
            dropout: Dropout probability
            batch_first: Whether batch dimension is first
            masking_p: Probability of masking tokens
            n_tokens: Total number of tokens (excluding auxiliary)
            context_length: Length of the context window
            lr: Learning rate
            warmup: Number of warmup steps
            batch_size: Batch size
            max_epochs: Maximum number of epochs
            cls_classes: Number of classification classes
            supervised_task: Type of supervised task
            learnable_pe: Whether to use learnable positional embeddings
            specie: Whether to add specie token
            assay: Whether to add assay token
            modality: Whether to add modality token
            contrastive: Whether to use contrastive loss
        """
        super().__init__()
        self.save_hyperparameters()

        # Core transformer components
        self._init_transformer(dim_model, nheads, dim_feedforward, nlayers, dropout, batch_first)

        # Embedding layers
        self._init_embeddings(n_tokens, dim_model, context_length, learnable_pe, dropout)

        # Task-specific heads
        self._init_heads(dim_model, n_tokens)

        # Loss functions
        self.loss = nn.CrossEntropyLoss()
        if supervised_task is not None:
            self.cls_loss = nn.CrossEntropyLoss()

        # Training metrics
        self.gc_freq = 5
        self.batch_train_losses = []

        self.initialize_weights()

    def _init_transformer(self, dim_model: int, nheads: int, dim_feedforward: int, 
                         nlayers: int, dropout: float, batch_first: bool) -> None:
        """Initialize transformer components."""
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim_model,
            nhead=nheads,
            dim_feedforward=dim_feedforward,
            batch_first=batch_first,
            dropout=dropout,
            layer_norm_eps=1e-12
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer=self.encoder_layer,
            num_layers=nlayers,
            enable_nested_tensor=False
        )

    def _init_embeddings(self, n_tokens: int, dim_model: int, context_length: int,
                        learnable_pe: bool, dropout: float) -> None:
        """Initialize embedding layers."""
        self.embeddings = nn.Embedding(
            num_embeddings=n_tokens+5,
            embedding_dim=dim_model,
            padding_idx=1
        )

        if learnable_pe:
            self.positional_embedding = nn.Embedding(
                num_embeddings=context_length,
                embedding_dim=dim_model
            )
            self.dropout = nn.Dropout(p=dropout)
            #self.register_buffer('pos', torch.arange(0, context_length, dtype=torch.long))
            self.pos = torch.arange(0, context_length, dtype=torch.long)
        else:
            self.positional_embedding = PositionalEncoding(
                d_model=dim_model,
                max_seq_len=context_length
            )

    def _init_heads(self, dim_model: int, n_tokens: int) -> None:
        """Initialize model heads."""
        self.classifier_head = nn.Linear(dim_model, n_tokens, bias=False)
        self.classifier_head.bias = nn.Parameter(torch.zeros(n_tokens))

        self.pooler_head = nn.Linear(dim_model, dim_model)
        self.activation = nn.Tanh()
        self.cls_head = nn.Linear(dim_model, 164)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward pass of the model."""
        token_embedding = self.embeddings(x)

        if self.hparams.learnable_pe:
            pos_embedding = self.positional_embedding(self.pos.to(token_embedding.device))
            embeddings = self.dropout(token_embedding + pos_embedding)
        else:
            embeddings = self.positional_embedding(token_embedding)

        transformer_output = self.encoder(
            embeddings,
            is_causal=False,
            src_key_padding_mask=attention_mask
        )

        prediction = self.classifier_head(transformer_output)

        return {
            'mlm_prediction': prediction,
            'transformer_output': transformer_output
        }

    def training_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        """Execute training step."""
        with torch.no_grad():
            batch = complete_masking(batch, self.hparams.masking_p, self.hparams.n_tokens)

        masked_indices = batch['masked_indices']
        mask = batch['mask']
        real_indices = batch['X']
        attention_mask = batch['attention_mask']

        predictions = self.forward(masked_indices, attention_mask)
        mlm_predictions = predictions['mlm_prediction']

        real_indices = self._prepare_target_indices(mask, real_indices)
        loss = self._compute_loss(mlm_predictions, real_indices)

        self.log('train_loss', loss, sync_dist=True, prog_bar=True, reduce_fx='mean')
        return loss

    def validation_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        """Execute validation step."""
        with torch.no_grad():
            batch = complete_masking(batch, self.hparams.masking_p, self.hparams.n_tokens)

        masked_indices = batch['masked_indices']
        mask = batch['mask']
        real_indices = batch['X']
        attention_mask = batch['attention_mask']

        predictions = self.forward(masked_indices, attention_mask)
        mlm_predictions = predictions['mlm_prediction']

        real_indices = self._prepare_target_indices(mask, real_indices)
        loss = self._compute_loss(mlm_predictions, real_indices)

        self.log('val_loss', loss, sync_dist=True, prog_bar=True, reduce_fx='mean')
        return loss

    def _apply_masking(self, batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        """Apply masking to input batch."""
        return complete_masking(batch, self.hparams.masking_p, self.hparams.n_tokens)

    def _prepare_target_indices(self, mask: torch.Tensor, real_indices: torch.Tensor) -> torch.Tensor:
        """Prepare target indices for loss computation. Only masked indices are taken into account."""
        return torch.where(
            mask == MASK_TOKEN,
            real_indices,
            torch.tensor(-100, dtype=torch.long, device=real_indices.device)
        ).type(torch.int64)

    def _compute_loss(self, predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """Compute loss for training/validation."""
        predictions = predictions.view(-1, self.hparams.n_tokens)
        targets = targets.view(-1)

        if self.hparams.masking_p == 0.0:
            return torch.tensor(0.0, device=predictions.device)
        return self.loss(predictions, targets)

    def _apply_positional_encoding(self, token_embedding: torch.Tensor) -> torch.Tensor:
        """Apply positional encoding to token embeddings."""
        if self.hparams.learnable_pe:
            pos_embedding = self.positional_embedding(self.pos.to(token_embedding.device))
            return self.dropout(token_embedding + pos_embedding)
        return self.positional_embedding(token_embedding)


    def on_before_batch_transfer(self, batch: tuple, dataloader_idx: int) -> dict[str, torch.Tensor]:
        """Process batch before device transfer."""
        if isinstance(batch, tuple):
            batch, _ = batch
        return batch

    def on_after_batch_transfer(self, batch: dict[str, torch.Tensor], 
                              dataloader_idx: int) -> dict[str, torch.Tensor]:
        """Process batch after device transfer."""
        data_key = 'X'
        x = batch[data_key]

        if self.hparams.modality and 'modality' in batch:
            x = torch.cat((batch['modality'].reshape(-1, 1).to(torch.int32), x), dim=1)

        if self.hparams.assay and 'assay' in batch:
            x = torch.cat((batch['assay'].reshape(-1, 1).to(torch.int32), x), dim=1)

        if self.hparams.specie and 'specie' in batch:
            x = torch.cat((batch['specie'].reshape(-1, 1).to(torch.int32), x), dim=1)

        if self.hparams.supervised_task:
            if 'cell_type' in batch:
                batch['label'] = batch['cell_type']
            if 'X_niche' in batch:
                batch['label'] = batch['X_niche']

        batch[data_key] = x[:, :self.hparams.context_length]
        return batch

    def configure_optimizers(self) -> tuple:
        """Configure optimizer and learning rate scheduler."""
        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.1)
        scheduler = CosineWarmupScheduler(
            optimizer,
            warmup=self.hparams.warmup,
            max_epochs=self.hparams.max_epochs
        )

        return [optimizer], [{'scheduler': scheduler, 'interval': 'step'}]

    def initialize_weights(self) -> None:
        """Initialize model weights."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.xavier_normal_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)

    def get_embeddings(self, batch: dict[str, torch.Tensor], layer: int = -1, with_context: bool = False) -> torch.Tensor:
        """Get embeddings from the model."""
        
        batch = self.on_after_batch_transfer(batch, 0)
        
        batch = complete_masking(batch, 0.0, self.hparams.n_tokens + 5)
        masked_indices = batch['masked_indices']
        attention_mask = batch['attention_mask']

        # Get token embeddings and positional encodings
        token_embedding = self.embeddings(masked_indices)
        
        if self.hparams.learnable_pe:
            pos_embedding = self.positional_embedding(self.pos.to(token_embedding.device))
            embeddings = self.dropout(token_embedding + pos_embedding)
        else:
            embeddings = self.positional_embedding(token_embedding)

        # Process through transformer layers up to desired layer
        if layer < 0:
            layer = len(self.encoder.layers) + layer # -1 means last layer

        for i in range(layer + 1):
            embeddings = self.encoder.layers[i](
                embeddings,
                src_key_padding_mask=attention_mask,
                is_causal=False
            )

        if not with_context:
            embeddings = embeddings[:, 3:, :]

        embeddings = embeddings.mean(dim=1)

        return embeddings


class PositionalEncoding(nn.Module):
    """Positional encoding using sine and cosine functions."""

    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        encoding = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term)
        encoding = encoding.unsqueeze(0)

        self.register_buffer('encoding', encoding, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input tensor."""
        return x + self.encoding[:, :x.size(1)]


class CosineWarmupScheduler(optim.lr_scheduler._LRScheduler):
    """Learning rate scheduler with cosine annealing and warmup."""
    
    def __init__(self, optimizer: optim.Optimizer, warmup: int, max_epochs: int):
        self.warmup = warmup
        self.max_num_epochs = max_epochs
        super().__init__(optimizer)

    def get_lr(self) -> list[float]:
        """Get learning rates for all parameter groups."""
        lr_factor = self.get_lr_factor(epoch=self.last_epoch)
        return [max(1e-5, base_lr * lr_factor) for base_lr in self.base_lrs]

    def get_lr_factor(self, epoch: int) -> float:
        """Calculate learning rate factor based on epoch."""
        lr_factor = 0.5 * (1 + np.cos(np.pi * epoch / self.max_num_epochs))
        if epoch <= self.warmup:
            lr_factor *= epoch * 1.0 / self.warmup
        return lr_factor


# Running the original Nicheformer
---------------------
Runs inference with the Python code above. The result (`nf_embeddings`) is the reference for the TorchScript and C++ versions.

Required files in this folder:
- `nicheformer.ckpt`: pretrained weights. The file is 572.2 MB and is not in the repo; download it as described in the original Nicheformer GitHub repo.
- `model.h5ad`: reference AnnData that defines the gene set and gene order expected by the model.
- `cosmx_mean_script.npy`: per-gene technology mean for CosMx, used for normalization.
- `spe_converted.h5ad`: input data, with raw counts in `layers['counts']`.

The checkpoint is loaded with `weights_only=False` because it contains NumPy objects that PyTorch >= 2.6 rejects by default. Only do this with a trusted checkpoint.

In [7]:
import os
import numpy as np
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader
import anndata as ad
from typing import Optional, Dict, Any
from tqdm import tqdm

config = {
    'data_path': './spe_converted.h5ad', #'path/to/your/data.h5ad',  # Path to your AnnData file
    'technology_mean_path': './cosmx_mean_script.npy', #'path/to/technology_mean.npy',  # Path to technology mean file
    'checkpoint_path': './nicheformer.ckpt',  # Path to model checkpoint
    'output_path': 'data_with_embeddings.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '.',  # Directory for any intermediate outputs
    'batch_size': 4,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 1000, # to prevent OOM
    'num_workers': 4,
    'precision': 32,
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

In [8]:
# Reference AnnData: defines the gene set and order. Note: `model` is overwritten later by the Nicheformer model.
model = ad.read_h5ad('./model.h5ad')

# Load data
adata = ad.read_h5ad(config['data_path'])
adata.X = adata.layers['counts']  # the model expects raw counts
technology_mean = np.load(config['technology_mean_path'])

# Align genes with the reference: outer join adds the missing reference genes (NaN/0, set to 0 during tokenization)
adata = ad.concat([model, adata], join='outer', axis=0)
# Drop the reference cell (row 0) and keep only the reference genes, in reference order
adata = adata[1:, model.var_names].copy()

# Context token IDs used by the pretrained model
adata.obs['modality'] = 4  # spatial
adata.obs['specie'] = 5    # human
adata.obs['assay'] = 8     # CosMx

In [9]:
# Package equivalent (not needed, classes defined above): from nicheformer.data import NicheformerDataset

# NicheformerDataset only keeps cells whose nicheformer_split == split, so all cells are labelled 'train'
adata.obs['nicheformer_split'] = 'train'

# Create dataset
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,  # must match the model context_length (1500)
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000),
    metadata_fields={'obs': ['modality', 'specie', 'assay']}
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

100%|██████████| 1/1 [00:02<00:00,  2.79s/it]


In [11]:
# Load the pretrained model (weights_only=False: see note above)
model = Nicheformer.load_from_checkpoint(
    checkpoint_path=config['checkpoint_path'],
    strict=False,
    weights_only=False,
)

model.eval()  # Set to evaluation mode

Nicheformer(
  (encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
    )
    (linear1): Linear(in_features=512, out_features=1024, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (linear2): Linear(in_features=1024, out_features=512, bias=True)
    (norm1): LayerNorm((512,), eps=1e-12, elementwise_affine=True, bias=True)
    (norm2): LayerNorm((512,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout1): Dropout(p=0.0, inplace=False)
    (dropout2): Dropout(p=0.0, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=

In [12]:
print("Extracting embeddings...")
embeddings = []
device = model.embeddings.weight.device

with torch.no_grad():
    for batch in tqdm(dataloader):
        # Move batch to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}

        # Get embeddings from the model
        emb = model.get_embeddings(
            batch=batch,
            layer=config.get('embedding_layer', -1)  # Default to last layer
        )
        embeddings.append(emb.cpu().numpy())


# Reference result: [n_cells, 512]
nf_embeddings = np.concatenate(embeddings, axis=0)



Extracting embeddings...


100%|██████████| 25/25 [00:15<00:00,  1.57it/s]


# TorchScript export
------------------------------
`NicheformerEmbedder` is a minimal, inference-only wrapper that reproduces `Nicheformer.get_embeddings` and can be compiled with `torch.jit.script`. It reuses the loaded weights; it does not copy them.

This section writes two files:
- `nicheformer_embedder.pt`: the compiled model. Load it with `torch.jit.load` (Python), `torch::jit::load` (C++) or `jit_load` (R).
- `tokens.pt`: the tokenized dataset (`X`, `modality`, `specie`, `assay`). It is the input for the C++ and R versions.

Notes:
- The wrapper only supports checkpoints with `learnable_pe=True`, and it omits dropout (eval mode only).
- With `with_context=False`, the first 3 positions (the specie, assay and modality tokens) are dropped before mean pooling, as in the original.

In [13]:
from typing import Dict

class NicheformerEmbedder(nn.Module):
    """TorchScript-friendly inference wrapper around a loaded Nicheformer."""
    def __init__(self, nf, layer: int = -1, with_context: bool = False):
        super().__init__()
        hp = nf.hparams                      # read hparams once, store as plain attributes
        self.embeddings = nf.embeddings      # shares the loaded weights
        self.positional_embedding = nf.positional_embedding
        self.layers = nf.encoder.layers
        n = len(nf.encoder.layers)
        self.n_layers = (n + layer if layer < 0 else layer) + 1
        self.with_context = with_context
        self.use_modality = bool(hp.modality)
        self.use_assay = bool(hp.assay)
        self.use_specie = bool(hp.specie)
        self.context_length = int(hp.context_length)
        self.register_buffer('pos', torch.arange(0, self.context_length, dtype=torch.long), persistent=False)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = batch['X']
        # same prepend order as on_after_batch_transfer
        if self.use_modality and 'modality' in batch:
            x = torch.cat((batch['modality'].reshape(-1, 1).to(torch.int32), x), dim=1)
        if self.use_assay and 'assay' in batch:
            x = torch.cat((batch['assay'].reshape(-1, 1).to(torch.int32), x), dim=1)
        if self.use_specie and 'specie' in batch:
            x = torch.cat((batch['specie'].reshape(-1, 1).to(torch.int32), x), dim=1)
        x = x[:, :self.context_length]

        # complete_masking(p=0) reduces to: 0 -> padding token 1, mask padding
        x = torch.where(x == 0, torch.ones_like(x), x)
        attention_mask = x == 1

        h = self.embeddings(x) + self.positional_embedding(self.pos[: x.shape[1]])
        i = 0
        for layer in self.layers:
            if i < self.n_layers:
                h = layer(h, src_key_padding_mask=attention_mask)
            i += 1
        if not self.with_context:
            h = h[:, 3:, :]
        return h.mean(dim=1)


In [14]:
embedder = torch.jit.script(NicheformerEmbedder(model, layer=config.get('embedding_layer', -1)).eval())
embedder.save('nicheformer_embedder.pt')   # loadable from Python, C++ and R

# Stack the whole dataset into one dict of tensors: X [n_cells, 1500], modality/specie/assay [n_cells]
items = [dataset[i] for i in range(len(dataset))]
tokens = {k: torch.stack([torch.as_tensor(it[k]) for it in items]) for k in items[0]}
torch.save(tokens, 'tokens.pt')  

embeddings = []

embedder = embedder.to(device)

In [15]:
%%time

import torch.jit as jit

# Reload the saved file to check that the serialized model gives the same result
device = "cuda"  # requires a GPU; use "cpu" otherwise
embedder = jit.load('./nicheformer_embedder.pt').eval().to(device)

embeddings = []
embedder = embedder.to(device)

with torch.inference_mode():
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        emb = embedder(batch)
        embeddings.append(emb.cpu().numpy())
        
# Should match nf_embeddings
jit_embeddings = np.concatenate(embeddings, axis=0)


CPU times: user 16.8 s, sys: 125 ms, total: 16.9 s
Wall time: 16.9 s


# Comparing original and compiled embeddings
-----------------
Shape and mean are used as a quick check; both should match.

## Compiled version

In [16]:
jit_embeddings.shape, jit_embeddings.mean()

np.float32(0.018332135)

## Original Nicheformer

In [17]:
nf_embeddings.shape, nf_embeddings.mean()

np.float32(0.018332135)

# Running the C++ version
-----------------------
Builds `../C/nicheformer.cpp` against the libtorch bundled with the installed PyTorch, then runs it on `tokens.pt`.

Arguments: compiled model, tokens, output file, batch size. Uses CUDA if available, otherwise CPU.

Notes:
- The CMake warnings about `CMAKE_CUDA_ARCHITECTURES` and `kineto` are harmless.
- On small GPUs, the run can fail with `CUDA error: out of memory` if this notebook still holds GPU memory. Move `model` and `embedder` to CPU and call `torch.cuda.empty_cache()` first, or prefix the command with `CUDA_VISIBLE_DEVICES=""` to run on CPU.

In [29]:
%%bash

cmake -S ../C -B build -DCMAKE_PREFIX_PATH="$(python3 -c 'import torch; print(torch.utils.cmake_prefix_path)')"
cmake --build build -j
./build/nicheformer_embed nicheformer_embedder.pt tokens.pt cpp_embeddings.pt 4   # last arg = batch size


-- PyTorch: CUDA detected: 12.6
-- PyTorch: CUDA nvcc is: /usr/local/cuda/bin/nvcc
-- PyTorch: CUDA toolkit directory: /usr/local/cuda
-- PyTorch: Header version is: 12.6
-- /usr/local/cuda/lib64/libnvrtc.so shorthash is a1f1e7d6
-- USE_CUDNN is set to 0. Compiling without cuDNN support
-- USE_CUSPARSELT is set to 0. Compiling without cuSPARSELt support
-- USE_CUDSS is set to 0. Compiling without cuDSS support
-- USE_CUFILE is set to 0. Compiling without cuFile support
-- Autodetected CUDA architecture(s):  8.9


CMake Warning at /opt/venv/lib/python3.12/site-packages/torch/share/cmake/Caffe2/public/cuda.cmake:332 (message):
  pytorch is not compatible with `CMAKE_CUDA_ARCHITECTURES` and will ignore
  its value.  Please configure `TORCH_CUDA_ARCH_LIST` instead.
Call Stack (most recent call first):
  /opt/venv/lib/python3.12/site-packages/torch/share/cmake/Caffe2/Caffe2Config.cmake:86 (include)
  /opt/venv/lib/python3.12/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:62 (find_package)
  CMakeLists.txt:4 (find_package)




-- Added CUDA NVCC flags for: -gencode;arch=compute_89,code=sm_89


CMake Warning at /opt/venv/lib/python3.12/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  library kineto not found.
Call Stack (most recent call first):
  /opt/venv/lib/python3.12/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:119 (append_torchlib_if_found)
  CMakeLists.txt:4 (find_package)




-- Configuring done (0.9s)
-- Generating done (0.0s)
-- Build files have been written to: /workspaces/project/nicheformer/Python/build
[100%] Built target nicheformer_embed
device: cuda
cells: 100, tokens per cell: 1500
embeddings: [100, 512]
saved cpp_embeddings.pt


# Loading the C++ results in Python
-----------------------
The mean should match the compiled and original versions above. The embeddings are stored in `adata.obsm['X_nicheformer_embeddings']`.

In [30]:
emb = torch.load('cpp_embeddings.pt')  # [n_cells, 512], written by the C++ program
adata.obsm['X_nicheformer_embeddings'] = emb.numpy()


In [31]:
adata.obsm['X_nicheformer_embeddings'].mean()

np.float32(0.018332135)